# Concept Bottleneck Models (ICML 2020) — Reproduction on CUB-200-2011

This notebook reproduces the main experiments from:
> **Concept Bottleneck Models**  
> Pang Wei Koh, Thao Nguyen, Yew Siang Tang, Stephen Mussmann, Emma Pierson, Been Kim, Percy Liang  
> ICML 2020 | [Paper](https://arxiv.org/abs/2007.04612) | [Code](https://github.com/yewsiang/ConceptBottleneck)

**Dataset**: CUB-200-2011 — 200 bird species, 112 binary concept attributes  
**Backbone**: InceptionV3 (pretrained on ImageNet)  

| # | Model | Description |
|---|-------|-------------|
| 1 | **X→C** (Independent) | Predict 112 concepts from images |
| 2 | **C→Y** (Oracle) | Predict species from ground-truth concepts |
| 3 | **Independent / Sequential** | Chain X→C then C→Y at test time |
| 4 | **Joint (X→C→Y)** | End-to-end concept bottleneck |
| 5 | **Standard (X→Y)** | Direct image-to-class baseline |
| 6 | **Multitask X→(C,Y)** | Joint prediction with auxiliary concept loss |

In [ ]:
# ============================================================
# CELL 1: Imports
# ============================================================
import os, sys, pickle, random, time, copy, json, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import defaultdict, OrderedDict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')

In [ ]:
# ============================================================
# CELL 2: Configuration
# ============================================================
# -------- MODE TOGGLE --------
# QUICK_MODE = True  → fast iteration  (15 epochs, ~1-2 h total on GPU)
# QUICK_MODE = False → full reproduction (1000 epochs w/ early stopping)
QUICK_MODE = False

# -------- Epoch & early-stopping --------
QUICK_EPOCHS   = 15
FULL_EPOCHS    = 1000
N_EPOCHS       = QUICK_EPOCHS if QUICK_MODE else FULL_EPOCHS
PATIENCE       = 5 if QUICK_MODE else 15

# -------- Hyperparameters (from paper / CodaLab) --------
BATCH_SIZE       = 64
LR               = 0.01
WEIGHT_DECAY     = 0.00004
SCHEDULER_STEP   = 1000        # effectively no LR decay in quick mode
ATTR_LOSS_WEIGHT = 0.01
N_ATTRIBUTES     = 112
N_CLASSES        = 200
SEED             = 1
NORMALIZE_LOSS   = True
USE_AUX          = True
PRETRAINED       = True

# -------- Paths (local) --------
BASE_DIR   = os.path.join(os.getcwd(), 'cub-200-2011')

DATA_DIR   = os.path.join(BASE_DIR, 'CUB_processed', 'class_attr_data_10')
IMAGE_DIR  = os.path.join(BASE_DIR, 'CUB_200_2011', 'images')
OUTPUT_DIR = os.path.join(os.getcwd(), 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------- Reproducibility --------
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Mode     : {"QUICK" if QUICK_MODE else "FULL"}')
print(f'Epochs   : {N_EPOCHS}   Patience: {PATIENCE}')
print(f'Device   : {DEVICE}')
print(f'Data dir : {DATA_DIR}')
print(f'Image dir: {IMAGE_DIR}')

# Global results dictionary
ALL_RESULTS = OrderedDict()

In [ ]:
# ============================================================
# CELL 3: Dataset & Data-Loading Utilities
# ============================================================

class CUBDataset(Dataset):
    '''CUB-200-2011 dataset for Concept Bottleneck Models.'''

    def __init__(self, pkl_paths, use_attr=True, no_img=False,
                 uncertain_label=False, image_dir='images',
                 n_class_attr=2, transform=None):
        self.data = []
        self.is_train = any('train' in p for p in pkl_paths)
        for path in pkl_paths:
            self.data.extend(pickle.load(open(path, 'rb')))
        self.transform      = transform
        self.use_attr        = use_attr
        self.no_img          = no_img
        self.uncertain_label = uncertain_label
        self.image_dir       = image_dir
        self.n_class_attr    = n_class_attr

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        d = self.data[idx]

        # ---- C -> Y (no image) ----
        if self.no_img:
            attr_labels  = d['attribute_label']
            class_label  = d['class_label']
            if isinstance(attr_labels, list):
                attr_labels = torch.FloatTensor(attr_labels)
            return attr_labels, class_label

        # ---- Build image path ----
        img_path = d['img_path'].replace('\\', '/')
        try:
            parts   = img_path.split('/')
            cub_idx = parts.index('CUB_200_2011')
            # parts after CUB_200_2011/images/ -> class_dir/file.jpg
            relative = '/'.join(parts[cub_idx + 2:])
            img_path = os.path.join(self.image_dir, relative)
        except (ValueError, IndexError):
            img_path = os.path.join(self.image_dir, os.path.basename(img_path))

        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)

        class_label = d['class_label']

        if self.use_attr:
            attr_labels = d['attribute_label']
            if isinstance(attr_labels, list):
                attr_labels = torch.FloatTensor(attr_labels)
            return img, class_label, attr_labels
        return img, class_label


def get_transforms(resol=299):
    '''InceptionV3-compatible transforms (299x299).'''
    resized = int(resol * 256 / 224)  # 342
    train_t = transforms.Compose([
        transforms.ColorJitter(brightness=32/255, saturation=(0.5, 1.5)),
        transforms.RandomResizedCrop(resol),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ])
    test_t = transforms.Compose([
        transforms.Resize((resized, resized)),
        transforms.CenterCrop(resol),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ])
    return train_t, test_t


def find_class_imbalance(pkl_path, n_attributes):
    '''Compute pos_weight for each concept attribute (neg/pos ratio).'''
    data = pickle.load(open(pkl_path, 'rb'))
    pos = np.zeros(n_attributes)
    neg = np.zeros(n_attributes)
    for d in data:
        for i, a in enumerate(d['attribute_label']):
            if a == 1:
                pos[i] += 1
            else:
                neg[i] += 1
    weights = []
    for i in range(n_attributes):
        w = neg[i] / max(pos[i], 1)
        weights.append(w)
    return weights

print('Dataset utilities loaded.')

In [ ]:
# ============================================================
# CELL 4: Load Data
# ============================================================
train_pkl = os.path.join(DATA_DIR, 'train.pkl')
val_pkl   = os.path.join(DATA_DIR, 'val.pkl')
test_pkl  = os.path.join(DATA_DIR, 'test.pkl')

train_t, test_t = get_transforms()

# ---- With images (for X->C, Joint, Standard, Multitask) ----
train_data = CUBDataset([train_pkl], use_attr=True,  no_img=False,
                         image_dir=IMAGE_DIR, transform=train_t)
val_data   = CUBDataset([val_pkl],   use_attr=True,  no_img=False,
                         image_dir=IMAGE_DIR, transform=test_t)
test_data  = CUBDataset([test_pkl],  use_attr=True,  no_img=False,
                         image_dir=IMAGE_DIR, transform=test_t)

# Loaders without attributes (for Standard X->Y)
train_data_no_attr = CUBDataset([train_pkl], use_attr=False, no_img=False,
                                 image_dir=IMAGE_DIR, transform=train_t)
val_data_no_attr   = CUBDataset([val_pkl],   use_attr=False, no_img=False,
                                 image_dir=IMAGE_DIR, transform=test_t)
test_data_no_attr  = CUBDataset([test_pkl],  use_attr=False, no_img=False,
                                 image_dir=IMAGE_DIR, transform=test_t)

# ---- Without images (for C->Y) ----
train_data_no_img = CUBDataset([train_pkl], use_attr=True, no_img=True)
val_data_no_img   = CUBDataset([val_pkl],   use_attr=True, no_img=True)
test_data_no_img  = CUBDataset([test_pkl],  use_attr=True, no_img=True)

NUM_WORKERS = 0  # 0 for Windows; increase on Linux

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

train_loader_no_attr = DataLoader(train_data_no_attr, batch_size=BATCH_SIZE,
                                  shuffle=True, num_workers=NUM_WORKERS,
                                  pin_memory=True, drop_last=True)
val_loader_no_attr   = DataLoader(val_data_no_attr,   batch_size=BATCH_SIZE,
                                  shuffle=False, num_workers=NUM_WORKERS,
                                  pin_memory=True)
test_loader_no_attr  = DataLoader(test_data_no_attr,  batch_size=BATCH_SIZE,
                                  shuffle=False, num_workers=NUM_WORKERS,
                                  pin_memory=True)

train_loader_no_img = DataLoader(train_data_no_img, batch_size=BATCH_SIZE,
                                 shuffle=True)
val_loader_no_img   = DataLoader(val_data_no_img,   batch_size=BATCH_SIZE,
                                 shuffle=False)
test_loader_no_img  = DataLoader(test_data_no_img,  batch_size=BATCH_SIZE,
                                 shuffle=False)

# Class-imbalance weights for weighted BCE
IMBALANCE_WEIGHTS = find_class_imbalance(train_pkl, N_ATTRIBUTES)

print(f'Train : {len(train_data):>5}  |  Val : {len(val_data):>5}  |  Test : {len(test_data):>5}')
print(f'Imbalance weight range: [{min(IMBALANCE_WEIGHTS):.2f}, {max(IMBALANCE_WEIGHTS):.2f}]')

In [ ]:
# ============================================================
# CELL 5: Model Definitions
# ============================================================

class ConceptBottleneckModel(nn.Module):
    '''
    Modified InceptionV3 for concept bottleneck models.
    Supports:
      - bottleneck=True,  n_attributes>0  → X->C  (concept-only output)
      - bottleneck=False, n_attributes>0  → X->(C,Y) or Joint
      - bottleneck=False, n_attributes=0  → X->Y  (standard baseline)
    '''
    def __init__(self, n_classes=200, n_attributes=112,
                 bottleneck=True, use_aux=True, pretrained=True,
                 freeze=False):
        super().__init__()
        self.n_attributes = n_attributes
        self.n_classes    = n_classes
        self.bottleneck   = bottleneck
        self.use_aux      = use_aux

        # --- Load InceptionV3 backbone ---
        try:
            self.backbone = models.inception_v3(
                weights='IMAGENET1K_V1', aux_logits=use_aux)
        except TypeError:
            self.backbone = models.inception_v3(
                pretrained=pretrained, aux_logits=use_aux)

        # Feature dimensions
        self.feat_dim = self.backbone.fc.in_features   # 2048
        self.backbone.fc = nn.Identity()

        # --- Main heads ---
        self.all_fc = nn.ModuleList()
        if not bottleneck:
            self.all_fc.append(nn.Linear(self.feat_dim, n_classes))
        for _ in range(n_attributes):
            self.all_fc.append(nn.Linear(self.feat_dim, 1))

        # --- Aux heads (only when aux_logits is used) ---
        if use_aux and hasattr(self.backbone, 'AuxLogits') and self.backbone.AuxLogits is not None:
            aux_dim = self.backbone.AuxLogits.fc.in_features  # 768
            self.backbone.AuxLogits.fc = nn.Identity()
            self.aux_fc = nn.ModuleList()
            if not bottleneck:
                self.aux_fc.append(nn.Linear(aux_dim, n_classes))
            for _ in range(n_attributes):
                self.aux_fc.append(nn.Linear(aux_dim, 1))

        if freeze:
            for param in self.backbone.parameters():
                param.requires_grad = False

    def forward(self, x):
        if self.training and self.use_aux:
            inc_out      = self.backbone(x)
            features     = inc_out.logits       # N x 2048
            aux_features = inc_out.aux_logits    # N x 768
            outputs      = [fc(features)     for fc in self.all_fc]
            aux_outputs  = [fc(aux_features) for fc in self.aux_fc]
            return outputs, aux_outputs
        else:
            features = self.backbone(x)          # N x 2048
            outputs  = [fc(features) for fc in self.all_fc]
            return outputs


class MLP(nn.Module):
    '''Simple MLP for C -> Y prediction.'''
    def __init__(self, input_dim, n_classes, expand_dim=0):
        super().__init__()
        if expand_dim:
            self.net = nn.Sequential(
                nn.Linear(input_dim, expand_dim),
                nn.ReLU(),
                nn.Linear(expand_dim, n_classes),
            )
        else:
            self.net = nn.Linear(input_dim, n_classes)

    def forward(self, x):
        return self.net(x)


class End2EndModel(nn.Module):
    '''Joint model: X -> C -> Y  (end-to-end concept bottleneck).'''
    def __init__(self, concept_model, class_model, use_sigmoid=False):
        super().__init__()
        self.concept_model = concept_model   # X -> C (bottleneck)
        self.class_model   = class_model     # C -> Y (MLP)
        self.use_sigmoid   = use_sigmoid
        self.use_aux       = concept_model.use_aux

    def _stage2(self, concept_outputs):
        '''Concepts -> class prediction.  Returns [class, c1, ..., cK].'''
        if self.use_sigmoid:
            processed = [torch.sigmoid(o) for o in concept_outputs]
        else:
            processed = concept_outputs
        stage2_in   = torch.cat(processed, dim=1)
        class_out   = self.class_model(stage2_in)
        return [class_out] + concept_outputs

    def forward(self, x):
        if self.training and self.use_aux:
            main, aux = self.concept_model(x)
            return self._stage2(main), self._stage2(aux)
        else:
            main = self.concept_model(x)
            return self._stage2(main)

print('Models defined: ConceptBottleneckModel, MLP, End2EndModel')

In [ ]:
# ============================================================
# CELL 6: Training & Evaluation Utilities
# ============================================================

def make_attr_criterion(imbalance_weights, device):
    '''Create per-attribute BCEWithLogitsLoss with class-imbalance weighting.'''
    criteria = []
    for w in imbalance_weights:
        pw = torch.FloatTensor([w]).to(device)
        criteria.append(nn.BCEWithLogitsLoss(pos_weight=pw))
    return criteria


# ---------- run_epoch for image-based models ----------
def run_epoch(model, optimizer, loader, criterion, attr_criterion,
              bottleneck, attr_loss_weight, normalize_loss, use_aux,
              is_training, use_attr=True):
    '''
    One epoch for image-based models:
      X->C, X->Y, X->(C,Y), Joint X->C->Y
    Returns (avg_loss, class_acc%, concept_acc%)
    '''
    if is_training:
        model.train()
    else:
        model.eval()

    total_loss  = 0.0
    n_samples   = 0
    cls_correct = 0
    cpt_correct = 0
    cpt_total   = 0

    for batch in loader:
        if use_attr:
            inputs, labels, attr_labels = batch
            attr_labels = attr_labels.float().to(DEVICE)
        else:
            inputs, labels = batch
            attr_labels = None

        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)
        bs = inputs.size(0)

        # --- forward ---
        if is_training and use_aux:
            outputs, aux_outputs = model(inputs)
        else:
            outputs = model(inputs)
            aux_outputs = None

        losses = []
        out_start = 0

        # --- class loss ---
        if not bottleneck:
            loss_cls = criterion(outputs[0], labels)
            if aux_outputs is not None:
                loss_cls = 1.0 * loss_cls + 0.4 * criterion(aux_outputs[0], labels)
            losses.append(loss_cls)
            out_start = 1
            _, preds = torch.max(outputs[0], 1)
            cls_correct += (preds == labels).sum().item()

        # --- concept losses ---
        if attr_criterion is not None and attr_loss_weight > 0 and attr_labels is not None:
            for i in range(len(attr_criterion)):
                c_loss = attr_criterion[i](
                    outputs[i + out_start].squeeze().float(),
                    attr_labels[:, i])
                if aux_outputs is not None:
                    c_loss = 1.0 * c_loss + 0.4 * attr_criterion[i](
                        aux_outputs[i + out_start].squeeze().float(),
                        attr_labels[:, i])
                losses.append(attr_loss_weight * c_loss)

            # concept accuracy
            for i in range(len(attr_criterion)):
                cpred = (torch.sigmoid(outputs[i + out_start].squeeze()) > 0.5).float()
                cpt_correct += (cpred == attr_labels[:, i]).sum().item()
                cpt_total   += bs

        # --- total loss ---
        batch_loss = sum(losses)
        if normalize_loss and attr_criterion is not None:
            batch_loss = batch_loss / N_ATTRIBUTES

        total_loss += batch_loss.item() * bs
        n_samples  += bs

        if is_training:
            optimizer.zero_grad()
            batch_loss.backward()
            optimizer.step()

    avg_loss   = total_loss / n_samples
    cls_acc    = cls_correct / n_samples * 100 if not bottleneck else 0.0
    cpt_acc    = cpt_correct / cpt_total * 100 if cpt_total > 0 else 0.0
    return avg_loss, cls_acc, cpt_acc


# ---------- run_epoch_simple for C->Y ----------
def run_epoch_simple(model, optimizer, loader, criterion, is_training):
    '''One epoch for attribute-only models (C -> Y).'''
    if is_training:
        model.train()
    else:
        model.eval()

    total_loss  = 0.0
    cls_correct = 0
    n_samples   = 0

    for inputs, labels in loader:
        if isinstance(inputs, list):
            inputs = torch.stack(inputs).t().float()
        inputs = inputs.float().to(DEVICE)
        labels = labels.to(DEVICE)
        bs = inputs.size(0)

        outputs = model(inputs)
        loss    = criterion(outputs, labels)

        total_loss += loss.item() * bs
        _, preds = torch.max(outputs, 1)
        cls_correct += (preds == labels).sum().item()
        n_samples   += bs

        if is_training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return total_loss / n_samples, cls_correct / n_samples * 100


# ---------- Generic training driver ----------
def train_image_model(model, train_ldr, val_ldr, n_epochs, lr, wd,
                      attr_criterion=None, bottleneck=False,
                      attr_loss_weight=0.01, normalize_loss=True,
                      use_aux=True, patience=15, model_name='model',
                      use_attr=True):
    '''Train an image-based model and return history dict.'''
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, momentum=0.9, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=SCHEDULER_STEP)

    history = {'train_loss': [], 'val_loss': [],
               'train_cls_acc': [], 'val_cls_acc': [],
               'train_cpt_acc': [], 'val_cpt_acc': []}
    best_val_loss = float('inf')
    wait = 0
    save_path = os.path.join(OUTPUT_DIR, f'{model_name}_best.pth')

    t0 = time.time()
    for epoch in range(n_epochs):
        tl, ta, tca = run_epoch(model, optimizer, train_ldr, criterion,
                                attr_criterion, bottleneck, attr_loss_weight,
                                normalize_loss, use_aux, True, use_attr)
        with torch.no_grad():
            vl, va, vca = run_epoch(model, None, val_ldr, criterion,
                                    attr_criterion, bottleneck, attr_loss_weight,
                                    normalize_loss, False, False, use_attr)
        scheduler.step()

        history['train_loss'].append(tl);   history['val_loss'].append(vl)
        history['train_cls_acc'].append(ta); history['val_cls_acc'].append(va)
        history['train_cpt_acc'].append(tca);history['val_cpt_acc'].append(vca)

        # --- early stopping ---
        if vl < best_val_loss:
            best_val_loss = vl
            wait = 0
            torch.save(model.state_dict(), save_path)
        else:
            wait += 1
            if wait >= patience:
                print(f'  >> Early stop @ epoch {epoch+1}')
                break

        if (epoch + 1) % max(1, n_epochs // 10) == 0 or epoch == 0:
            acc_str = f'ClsAcc {ta:5.1f}/{va:5.1f}' if not bottleneck else f'CptAcc {tca:5.1f}/{vca:5.1f}'
            print(f'  Ep {epoch+1:>4}/{n_epochs} | Loss {tl:.4f}/{vl:.4f} | {acc_str}')

    elapsed = time.time() - t0
    print(f'  Training finished in {elapsed/60:.1f} min')
    model.load_state_dict(torch.load(save_path, weights_only=True))
    return history


def train_simple_model(model, train_ldr, val_ldr, n_epochs, lr, wd,
                       patience=15, model_name='model'):
    '''Train a simple MLP (C -> Y) and return history.'''
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=SCHEDULER_STEP)

    history = {'train_loss': [], 'val_loss': [],
               'train_cls_acc': [], 'val_cls_acc': []}
    best_val_loss = float('inf')
    wait = 0
    save_path = os.path.join(OUTPUT_DIR, f'{model_name}_best.pth')

    # C->Y converges fast; use more epochs
    ctoy_epochs = n_epochs * 5 if QUICK_MODE else n_epochs

    t0 = time.time()
    for epoch in range(ctoy_epochs):
        tl, ta = run_epoch_simple(model, optimizer, train_ldr, criterion, True)
        with torch.no_grad():
            vl, va = run_epoch_simple(model, None, val_ldr, criterion, False)
        scheduler.step()

        history['train_loss'].append(tl);   history['val_loss'].append(vl)
        history['train_cls_acc'].append(ta); history['val_cls_acc'].append(va)

        if vl < best_val_loss:
            best_val_loss = vl
            wait = 0
            torch.save(model.state_dict(), save_path)
        else:
            wait += 1
            if wait >= patience * 2:   # more patience for small models
                print(f'  >> Early stop @ epoch {epoch+1}')
                break

        if (epoch + 1) % max(1, ctoy_epochs // 10) == 0 or epoch == 0:
            print(f'  Ep {epoch+1:>4}/{ctoy_epochs} | Loss {tl:.4f}/{vl:.4f} | Acc {ta:5.1f}/{va:5.1f}')

    elapsed = time.time() - t0
    print(f'  Training finished in {elapsed/60:.1f} min')
    model.load_state_dict(torch.load(save_path, weights_only=True))
    return history


# ---------- Evaluation helpers ----------
@torch.no_grad()
def evaluate_image_model(model, loader, attr_criterion, bottleneck,
                         attr_loss_weight, normalize_loss, use_attr=True):
    '''Evaluate on a data loader.  Returns (loss, cls_acc, cpt_acc).'''
    criterion = nn.CrossEntropyLoss()
    model.eval()
    return run_epoch(model, None, loader, criterion, attr_criterion,
                     bottleneck, attr_loss_weight, normalize_loss,
                     False, False, use_attr)


@torch.no_grad()
def evaluate_simple_model(model, loader):
    '''Evaluate a C->Y model.  Returns (loss, acc).'''
    criterion = nn.CrossEntropyLoss()
    model.eval()
    return run_epoch_simple(model, None, loader, criterion, False)


@torch.no_grad()
def get_concept_predictions(model, loader):
    '''Run X->C and collect (sigmoid_preds, true_labels, true_attrs).'''
    model.eval()
    all_preds, all_labels, all_attrs = [], [], []
    for inputs, labels, attr_labels in loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)                 # list of 112 tensors
        logits  = torch.cat(outputs, dim=1)     # N x 112
        preds   = torch.sigmoid(logits)
        all_preds.append(preds.cpu())
        all_labels.append(labels)
        all_attrs.append(attr_labels)
    return (torch.cat(all_preds),
            torch.cat(all_labels),
            torch.cat(all_attrs))


@torch.no_grad()
def compute_per_concept_accuracy(model, loader):
    '''Return per-concept binary accuracy (array of length N_ATTRIBUTES).'''
    model.eval()
    correct = np.zeros(N_ATTRIBUTES)
    total   = 0
    for inputs, labels, attr_labels in loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        attr_labels = attr_labels.numpy()
        for i in range(N_ATTRIBUTES):
            cpred = (torch.sigmoid(outputs[i].squeeze()).cpu().numpy() > 0.5).astype(float)
            correct[i] += (cpred == attr_labels[:, i]).sum()
        total += attr_labels.shape[0]
    return correct / total * 100   # shape (112,)

print('Training & evaluation utilities loaded.')

---
## Experiments
Each cell below trains / evaluates one model variant.  
You can run them independently or restart from any cell after a kernel timeout.

In [ ]:
# ============================================================
# EXPERIMENT 1: X -> C  (Independent Concept Predictor)
# ============================================================
print('=' * 60)
print('MODEL 1:  X -> C   (Independent Concept Predictor)')
print('=' * 60)

xtoc_model = ConceptBottleneckModel(
    n_classes=N_CLASSES, n_attributes=N_ATTRIBUTES,
    bottleneck=True, use_aux=USE_AUX, pretrained=PRETRAINED
).to(DEVICE)

attr_criterion = make_attr_criterion(IMBALANCE_WEIGHTS, DEVICE)

xtoc_history = train_image_model(
    xtoc_model, train_loader, val_loader,
    n_epochs=N_EPOCHS, lr=LR, wd=WEIGHT_DECAY,
    attr_criterion=attr_criterion, bottleneck=True,
    attr_loss_weight=ATTR_LOSS_WEIGHT, normalize_loss=NORMALIZE_LOSS,
    use_aux=USE_AUX, patience=PATIENCE, model_name='xtoc',
    use_attr=True)

# ---- Test set evaluation ----
test_loss, _, test_cpt_acc = evaluate_image_model(
    xtoc_model, test_loader, attr_criterion,
    bottleneck=True, attr_loss_weight=ATTR_LOSS_WEIGHT,
    normalize_loss=NORMALIZE_LOSS, use_attr=True)

per_concept_acc = compute_per_concept_accuracy(xtoc_model, test_loader)

print(f'\n>>> X->C  Test Concept Acc : {test_cpt_acc:.2f}%')
print(f'>>> X->C  Mean per-concept : {per_concept_acc.mean():.2f}%')

ALL_RESULTS['X→C'] = {
    'history': xtoc_history,
    'test_concept_acc': test_cpt_acc,
    'per_concept_acc': per_concept_acc,
}

In [ ]:
# ============================================================
# EXPERIMENT 2: C -> Y  (Oracle: ground-truth concepts → class)
# ============================================================
print('=' * 60)
print('MODEL 2:  C -> Y   (Oracle Concept-to-Class MLP)')
print('=' * 60)

oracle_ctoy = MLP(input_dim=N_ATTRIBUTES, n_classes=N_CLASSES,
                  expand_dim=0).to(DEVICE)

ctoy_history = train_simple_model(
    oracle_ctoy, train_loader_no_img, val_loader_no_img,
    n_epochs=N_EPOCHS, lr=LR, wd=WEIGHT_DECAY,
    patience=PATIENCE, model_name='oracle_ctoy')

# Test on ground-truth concepts
test_loss_gt, test_acc_gt = evaluate_simple_model(oracle_ctoy, test_loader_no_img)
print(f'\n>>> C->Y (ground-truth concepts) Test Acc : {test_acc_gt:.2f}%')

ALL_RESULTS['C→Y (oracle)'] = {
    'history': ctoy_history,
    'test_task_acc': test_acc_gt,
}

In [ ]:
# ============================================================
# EXPERIMENT 3: Independent & Sequential Evaluation
# ============================================================
print('=' * 60)
print('MODEL 3:  Independent & Sequential (chained X->C, C->Y)')
print('=' * 60)

# --- 3a: Get concept predictions from X->C on train/val/test ---
print('\nGenerating concept predictions from X->C ...')
train_cpreds, train_labels_t, train_attrs = get_concept_predictions(xtoc_model, train_loader)
val_cpreds,   val_labels_t,   val_attrs   = get_concept_predictions(xtoc_model, val_loader)
test_cpreds,  test_labels_t,  test_attrs  = get_concept_predictions(xtoc_model, test_loader)
print(f'  Train: {train_cpreds.shape}, Val: {val_cpreds.shape}, Test: {test_cpreds.shape}')

# --- 3b: INDEPENDENT = Oracle C->Y evaluated on predicted concepts ---
print('\n--- Independent (oracle C->Y + predicted concepts) ---')
oracle_ctoy.eval()
test_pred_dataset = TensorDataset(test_cpreds, test_labels_t)
test_pred_loader  = DataLoader(test_pred_dataset, batch_size=BATCH_SIZE, shuffle=False)
_, indep_acc = evaluate_simple_model(oracle_ctoy, test_pred_loader)
print(f'>>> Independent Test Acc : {indep_acc:.2f}%')

ALL_RESULTS['Independent'] = {'test_task_acc': indep_acc}

# --- 3c: SEQUENTIAL = C->Y trained on predicted concepts ---
print('\n--- Sequential (C->Y trained on predicted concepts) ---')
seq_ctoy = MLP(input_dim=N_ATTRIBUTES, n_classes=N_CLASSES, expand_dim=0).to(DEVICE)

seq_train_ds  = TensorDataset(train_cpreds, train_labels_t)
seq_val_ds    = TensorDataset(val_cpreds,   val_labels_t)
seq_train_ldr = DataLoader(seq_train_ds, batch_size=BATCH_SIZE, shuffle=True)
seq_val_ldr   = DataLoader(seq_val_ds,   batch_size=BATCH_SIZE, shuffle=False)

seq_history = train_simple_model(
    seq_ctoy, seq_train_ldr, seq_val_ldr,
    n_epochs=N_EPOCHS, lr=LR, wd=WEIGHT_DECAY,
    patience=PATIENCE, model_name='seq_ctoy')

_, seq_acc = evaluate_simple_model(seq_ctoy, test_pred_loader)
print(f'\n>>> Sequential Test Acc : {seq_acc:.2f}%')

ALL_RESULTS['Sequential'] = {
    'history': seq_history,
    'test_task_acc': seq_acc,
}

In [ ]:
# ============================================================
# EXPERIMENT 4: Joint X -> C -> Y  (End-to-End Bottleneck)
# ============================================================
print('=' * 60)
print('MODEL 4:  Joint X -> C -> Y  (End-to-End Bottleneck)')
print('=' * 60)

# Stage 1: concept predictor (fresh)
joint_concept = ConceptBottleneckModel(
    n_classes=N_CLASSES, n_attributes=N_ATTRIBUTES,
    bottleneck=True, use_aux=USE_AUX, pretrained=PRETRAINED
).to(DEVICE)

# Stage 2: class predictor MLP
joint_mlp = MLP(input_dim=N_ATTRIBUTES, n_classes=N_CLASSES, expand_dim=0).to(DEVICE)

# Combined end-to-end model
joint_model = End2EndModel(joint_concept, joint_mlp, use_sigmoid=False).to(DEVICE)

attr_criterion_joint = make_attr_criterion(IMBALANCE_WEIGHTS, DEVICE)

joint_history = train_image_model(
    joint_model, train_loader, val_loader,
    n_epochs=N_EPOCHS, lr=LR, wd=WEIGHT_DECAY,
    attr_criterion=attr_criterion_joint, bottleneck=False,   # outputs[0] = class
    attr_loss_weight=ATTR_LOSS_WEIGHT, normalize_loss=NORMALIZE_LOSS,
    use_aux=USE_AUX, patience=PATIENCE, model_name='joint',
    use_attr=True)

test_loss, test_cls_acc, test_cpt_acc = evaluate_image_model(
    joint_model, test_loader, attr_criterion_joint,
    bottleneck=False, attr_loss_weight=ATTR_LOSS_WEIGHT,
    normalize_loss=NORMALIZE_LOSS, use_attr=True)

print(f'\n>>> Joint  Test Task Acc    : {test_cls_acc:.2f}%')
print(f'>>> Joint  Test Concept Acc : {test_cpt_acc:.2f}%')

ALL_RESULTS['Joint'] = {
    'history': joint_history,
    'test_task_acc': test_cls_acc,
    'test_concept_acc': test_cpt_acc,
}

In [ ]:
# ============================================================
# EXPERIMENT 5: Standard X -> Y  (No Concepts)
# ============================================================
print('=' * 60)
print('MODEL 5:  Standard X -> Y  (No Concept Bottleneck)')
print('=' * 60)

standard_model = ConceptBottleneckModel(
    n_classes=N_CLASSES, n_attributes=0,
    bottleneck=False, use_aux=USE_AUX, pretrained=PRETRAINED
).to(DEVICE)

std_history = train_image_model(
    standard_model, train_loader_no_attr, val_loader_no_attr,
    n_epochs=N_EPOCHS, lr=LR, wd=WEIGHT_DECAY,
    attr_criterion=None, bottleneck=False,
    attr_loss_weight=0, normalize_loss=False,
    use_aux=USE_AUX, patience=PATIENCE, model_name='standard',
    use_attr=False)

test_loss, test_cls_acc, _ = evaluate_image_model(
    standard_model, test_loader_no_attr, None,
    bottleneck=False, attr_loss_weight=0,
    normalize_loss=False, use_attr=False)

print(f'\n>>> Standard  Test Task Acc : {test_cls_acc:.2f}%')

ALL_RESULTS['Standard'] = {
    'history': std_history,
    'test_task_acc': test_cls_acc,
}

In [ ]:
# ============================================================
# EXPERIMENT 6: Multitask X -> (C, Y)
# ============================================================
print('=' * 60)
print('MODEL 6:  Multitask X -> (C, Y)  (Auxiliary Concept Loss)')
print('=' * 60)

multi_model = ConceptBottleneckModel(
    n_classes=N_CLASSES, n_attributes=N_ATTRIBUTES,
    bottleneck=False, use_aux=USE_AUX, pretrained=PRETRAINED
).to(DEVICE)

attr_criterion_multi = make_attr_criterion(IMBALANCE_WEIGHTS, DEVICE)

multi_history = train_image_model(
    multi_model, train_loader, val_loader,
    n_epochs=N_EPOCHS, lr=LR, wd=WEIGHT_DECAY,
    attr_criterion=attr_criterion_multi, bottleneck=False,
    attr_loss_weight=ATTR_LOSS_WEIGHT, normalize_loss=NORMALIZE_LOSS,
    use_aux=USE_AUX, patience=PATIENCE, model_name='multitask',
    use_attr=True)

test_loss, test_cls_acc, test_cpt_acc = evaluate_image_model(
    multi_model, test_loader, attr_criterion_multi,
    bottleneck=False, attr_loss_weight=ATTR_LOSS_WEIGHT,
    normalize_loss=NORMALIZE_LOSS, use_attr=True)

print(f'\n>>> Multitask  Test Task Acc    : {test_cls_acc:.2f}%')
print(f'>>> Multitask  Test Concept Acc : {test_cpt_acc:.2f}%')

ALL_RESULTS['Multitask'] = {
    'history': multi_history,
    'test_task_acc': test_cls_acc,
    'test_concept_acc': test_cpt_acc,
}

---
## Results & Visualizations

In [ ]:
# ============================================================
# RESULTS TABLE
# ============================================================
print('\n' + '=' * 70)
print('  CBM PAPER REPRODUCTION — RESULTS SUMMARY')
print('=' * 70)
print(f'{"Model":<20} {"Task Acc (%)":>14} {"Concept Acc (%)":>16}')
print('-' * 52)

# Paper's reported numbers for comparison (Table 1, CUB)
PAPER_RESULTS = {
    'Independent': 82.77,
    'Sequential':  82.77,
    'Joint':       84.26,
    'Standard':    81.63,
    'Multitask':   None,  # not directly in Table 1
}

for name, res in ALL_RESULTS.items():
    t_acc = res.get('test_task_acc', '-')
    c_acc = res.get('test_concept_acc', '-')
    t_str = f'{t_acc:>12.2f}%' if isinstance(t_acc, (int, float)) else f'{t_acc:>13}'
    c_str = f'{c_acc:>14.2f}%' if isinstance(c_acc, (int, float)) else f'{c_acc:>15}'
    print(f'{name:<20} {t_str} {c_str}')

print('\n--- Paper\'s reported numbers (CUB, Table 1) ---')
for name, acc in PAPER_RESULTS.items():
    if acc is not None:
        print(f'  {name:<20} {acc:.2f}%')

In [ ]:
# ============================================================
# GRAPH 1: Training & Validation Loss Curves
# ============================================================
models_with_history = {k: v for k, v in ALL_RESULTS.items() if 'history' in v}
n_models = len(models_with_history)

if n_models > 0:
    cols = min(3, n_models)
    rows = (n_models + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.5 * rows))
    if n_models == 1:
        axes = np.array([axes])
    axes = np.atleast_2d(axes)

    for idx, (name, res) in enumerate(models_with_history.items()):
        r, c = idx // cols, idx % cols
        ax = axes[r, c]
        h = res['history']
        epochs = range(1, len(h['train_loss']) + 1)
        ax.plot(epochs, h['train_loss'], color='#3498db', lw=1.5, label='Train')
        ax.plot(epochs, h['val_loss'],   color='#e74c3c', lw=1.5, label='Val')
        ax.set_title(name, fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

    # hide unused subplots
    for idx in range(n_models, rows * cols):
        axes[idx // cols, idx % cols].set_visible(False)

    fig.suptitle('Training & Validation Loss', fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'loss_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/loss_curves.png')

In [ ]:
# ============================================================
# GRAPH 2: Training & Validation Accuracy Curves
# ============================================================
if n_models > 0:
    cols = min(3, n_models)
    rows = (n_models + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.5 * rows))
    if n_models == 1:
        axes = np.array([axes])
    axes = np.atleast_2d(axes)

    for idx, (name, res) in enumerate(models_with_history.items()):
        r, c = idx // cols, idx % cols
        ax = axes[r, c]
        h = res['history']
        epochs = range(1, len(h['train_loss']) + 1)

        # Pick the right accuracy key
        if 'train_cpt_acc' in h and max(h.get('train_cpt_acc', [0])) > 0:
            tr_acc = h['train_cpt_acc']
            vl_acc = h['val_cpt_acc']
            ylabel = 'Concept Acc (%)'
        else:
            tr_acc = h['train_cls_acc']
            vl_acc = h['val_cls_acc']
            ylabel = 'Task Acc (%)'

        ax.plot(epochs, tr_acc, color='#2ecc71', lw=1.5, label='Train')
        ax.plot(epochs, vl_acc, color='#e67e22', lw=1.5, label='Val')
        ax.set_title(name, fontsize=12, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

    for idx in range(n_models, rows * cols):
        axes[idx // cols, idx % cols].set_visible(False)

    fig.suptitle('Training & Validation Accuracy', fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'accuracy_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/accuracy_curves.png')

In [ ]:
# ============================================================
# GRAPH 3: Test Accuracy Comparison Bar Chart
# ============================================================
task_models = {k: v for k, v in ALL_RESULTS.items() if 'test_task_acc' in v}

if task_models:
    fig, ax = plt.subplots(figsize=(10, 6))
    names = list(task_models.keys())
    accs  = [task_models[n]['test_task_acc'] for n in names]
    palette = ['#1abc9c', '#3498db', '#9b59b6', '#e74c3c', '#f39c12', '#2ecc71']
    colors  = palette[:len(names)]

    bars = ax.bar(names, accs, color=colors, edgecolor='white', linewidth=1.2, width=0.6)
    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)

    # Paper baselines (dashed lines)
    paper_vals = [v for v in PAPER_RESULTS.values() if v is not None]
    if paper_vals:
        ax.axhline(y=max(paper_vals), color='gray', ls='--', lw=1,
                   label=f'Paper best ({max(paper_vals):.1f}%)')
        ax.legend(fontsize=10)

    ax.set_ylabel('Test Accuracy (%)', fontsize=12)
    ax.set_title('CBM Paper Reproduction — Test Task Accuracy',
                 fontsize=14, fontweight='bold')
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'test_accuracy_bar.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/test_accuracy_bar.png')

In [ ]:
# ============================================================
# GRAPH 4: Per-Concept Accuracy Distribution (from X->C)
# ============================================================
if 'X→C' in ALL_RESULTS and 'per_concept_acc' in ALL_RESULTS['X→C']:
    pca = ALL_RESULTS['X→C']['per_concept_acc']

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # --- Histogram ---
    ax = axes[0]
    ax.hist(pca, bins=20, color='#3498db', edgecolor='white', alpha=0.85)
    ax.axvline(pca.mean(), color='#e74c3c', ls='--', lw=2,
               label=f'Mean = {pca.mean():.1f}%')
    ax.set_xlabel('Binary Accuracy (%)', fontsize=11)
    ax.set_ylabel('Number of Concepts', fontsize=11)
    ax.set_title('Per-Concept Accuracy Distribution', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)

    # --- Sorted bar ---
    ax2 = axes[1]
    sorted_acc = np.sort(pca)
    colors_bar = ['#e74c3c' if a < 80 else '#f39c12' if a < 90 else '#2ecc71'
                  for a in sorted_acc]
    ax2.bar(range(N_ATTRIBUTES), sorted_acc, color=colors_bar, width=1.0)
    ax2.axhline(pca.mean(), color='#e74c3c', ls='--', lw=1.5,
                label=f'Mean = {pca.mean():.1f}%')
    ax2.set_xlabel('Concept Index (sorted)', fontsize=11)
    ax2.set_ylabel('Accuracy (%)', fontsize=11)
    ax2.set_title('Per-Concept Accuracy (sorted)', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'concept_accuracy_dist.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/concept_accuracy_dist.png')
    print(f'\nConcept accuracy stats:')
    print(f'  Mean : {pca.mean():.2f}%')
    print(f'  Min  : {pca.min():.2f}%  (concept {np.argmin(pca)})')
    print(f'  Max  : {pca.max():.2f}%  (concept {np.argmax(pca)})')
    print(f'  <80% : {(pca < 80).sum()} concepts')
    print(f'  >95% : {(pca > 95).sum()} concepts')

In [ ]:
# ============================================================
# GRAPH 5: Comparison with Paper Results
# ============================================================
comparable = {}
for name in ['Independent', 'Sequential', 'Joint', 'Standard']:
    if name in ALL_RESULTS and 'test_task_acc' in ALL_RESULTS[name]:
        if name in PAPER_RESULTS and PAPER_RESULTS[name] is not None:
            comparable[name] = {
                'ours': ALL_RESULTS[name]['test_task_acc'],
                'paper': PAPER_RESULTS[name],
            }

if comparable:
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(comparable))
    w = 0.35
    names = list(comparable.keys())
    ours  = [comparable[n]['ours']  for n in names]
    paper = [comparable[n]['paper'] for n in names]

    b1 = ax.bar(x - w/2, ours,  w, label='Ours (reproduction)', color='#3498db',
                edgecolor='white', linewidth=1.2)
    b2 = ax.bar(x + w/2, paper, w, label='Paper (reported)',    color='#95a5a6',
                edgecolor='white', linewidth=1.2)

    for bar, val in zip(b1, ours):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    for bar, val in zip(b2, paper):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{val:.1f}', ha='center', va='bottom', fontsize=10, color='#555')

    ax.set_xticks(x)
    ax.set_xticklabels(names, fontsize=11)
    ax.set_ylabel('Test Accuracy (%)', fontsize=12)
    ax.set_title('Reproduction vs. Paper Results', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'reproduction_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/reproduction_comparison.png')
else:
    print('Not enough comparable results to plot.')

In [ ]:
# ============================================================
# GRAPH 6: Combined Training Loss Overlay
# ============================================================
if models_with_history:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    palette = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']

    # --- Loss overlay ---
    ax = axes[0]
    for idx, (name, res) in enumerate(models_with_history.items()):
        h = res['history']
        ep = range(1, len(h['val_loss']) + 1)
        ax.plot(ep, h['val_loss'], color=palette[idx % len(palette)],
                lw=1.8, label=name)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Validation Loss', fontsize=11)
    ax.set_title('Validation Loss Comparison', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

    # --- Accuracy overlay ---
    ax2 = axes[1]
    for idx, (name, res) in enumerate(models_with_history.items()):
        h = res['history']
        if 'val_cls_acc' in h and max(h.get('val_cls_acc', [0])) > 0:
            ep = range(1, len(h['val_cls_acc']) + 1)
            ax2.plot(ep, h['val_cls_acc'], color=palette[idx % len(palette)],
                     lw=1.8, label=name)
    ax2.set_xlabel('Epoch', fontsize=11)
    ax2.set_ylabel('Validation Task Acc (%)', fontsize=11)
    ax2.set_title('Validation Task Accuracy Comparison', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'combined_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/combined_comparison.png')

In [ ]:
# ============================================================
# SAVE ALL RESULTS TO JSON
# ============================================================
serializable = {}
for name, res in ALL_RESULTS.items():
    entry = {}
    for k, v in res.items():
        if k == 'per_concept_acc':
            entry[k] = v.tolist()
        elif k == 'history':
            entry[k] = {hk: [float(x) for x in hv] for hk, hv in v.items()}
        else:
            entry[k] = float(v) if isinstance(v, (int, float, np.floating)) else v
    serializable[name] = entry

with open(os.path.join(OUTPUT_DIR, 'all_results.json'), 'w') as f:
    json.dump(serializable, f, indent=2)

print('All results saved to outputs/all_results.json')
print('\n=== REPRODUCTION COMPLETE ===')